In [0]:
%run ./config

In [0]:
agg_df=spark.read.csv(silver+'/loan_repayments_agg.csv', header=True, inferSchema=True)

In [0]:
agg_df.createOrReplaceTempView('agg_df')

In [0]:
active_loans = agg_df.select(countDistinct("loan_id").alias("total_active_loans"))
active_loans.write.mode("overwrite").option("overwriteSchema", "true").format('delta').save(gold+'/active_loans')

In [0]:
total_outstanding_amount=spark.sql("""
                                with cte as (
                                select distinct loan_id, principal from agg_df
                                ),
                                cte2 as (
                                select loan_id, round(sum(amount_paid),2) amount_paid from agg_df group by loan_id
                                ),
                                cte3 as (
                                select c1.loan_id, c1.principal-c2.amount_paid outstanding_amount 
                                from cte c1 join cte2 c2 
                                on c1.loan_id = c2.loan_id
                                )

                                select round(sum(outstanding_amount),2) total_outstanding_amount from cte3""")
total_outstanding_amount.write.mode('overwrite').format('delta').save(gold+'/total_outstanding_amount')

In [0]:
avg_urs_score=spark.sql(""" 
                        with cte1 as (
                            select distinct loan_id, urs_score from agg_df 
                        )
                        select round(avg(urs_score),2) avg_urs_score from cte1
                        """)
avg_urs_score.write.mode('overwrite').format('delta').save(gold+'/avg_urs_score')

In [0]:
percentage_severely_overdue=spark.sql("""
                                      with cte1 as (
                                          select sum(case when days_past_due > 30 then 1 else 0 end) as total_severely_overdue_loans, 
                                          count(*) as total_loans 
                                          from agg_df 
                                      )
                                      select round(total_severely_overdue_loans*100/total_loans,2) as percentage_severely_overdue from cte1
                                      """)
percentage_severely_overdue.write.mode('overwrite').format('delta').save(gold+'/percentage_severely_overdue')

In [0]:
percentage_overdue_loans= spark.sql(""" 
                                    with cte1 as (
                                        select sum(case when repayment_status != 'on-time' 
                                        then 1 
                                        else 0 end) as total_overdue_loans, 
                                        count(*) as total_loans 
                                        from agg_df )
                                    select round((total_overdue_loans/total_loans)*100,2) as percentage_overdue_loans from cte1
                                    """)
percentage_overdue_loans.write.mode('overwrite').format('delta').save(gold+'/percentage_overdue_loans')